# Prépa TD02 — 3 perceptrons, puis une matrice
Au TD01, un `Perceptron` + Heaviside sépare le plan en deux (0/1).
Ici on veut **3 classes** : on prend **3 perceptrons**, chacun répond 0/1.
D'abord en **boucle Python** (Exo 1), puis rangés en **matrice `W`, vecteur `b`** (Exo 2) :
on doit obtenir exactement les mêmes réponses. Sans `argmax` pour l'instant :
la sortie est un **vecteur `[0/1, 0/1, 0/1]`**.

In [ ]:
# EXPORT
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

## Code fourni : `Perceptron` (TD01) + données
`Perceptron(w, b).output(x)` vaut 1 si `w·x + b >= 0`, sinon 0 (Heaviside).
`gen_blobs()` donne 75 points 2D en 3 classes. Rien à coder ici : exécutez.

In [ ]:
# EXPORT
@dataclass
class Perceptron:
    w: np.ndarray
    b: float
    def output(self, x: np.ndarray) -> int:
        return 1 if float(self.w @ x + self.b) >= 0 else 0

def gen_blobs(n=25, sigma=0.6, seed=0):
    rng = np.random.default_rng(seed)
    centres = [(-2,-2),(2,-2),(0,2)]
    X_list, y_list = [], []
    for k,(cx,cy) in enumerate(centres):
        pts = rng.normal(loc=(cx,cy), scale=sigma, size=(n,2))
        X_list.append(pts)
        y_list += [k]*n
    return np.vstack(X_list), np.array(y_list)

X_train, y_train = gen_blobs()
print(X_train.shape, sorted(set(y_train.tolist())))

## Exo 1 — boucle sur 3 perceptrons
3 perceptrons fournis ci-dessous (poids imposés, `b=-0.5` pour les trois).
Complétez `predict_liste(x, perceptrons) -> np.ndarray` : boucle sur `p.output(x)`,
retourne `np.array([0/1, 0/1, 0/1])`. Vérifiez avec la cellule suivante :
`[0,0]` doit donner `[0 0 0]`, `[0,2]` doit donner `[1 1 0]`.

In [ ]:
# EXPORT
import numpy as np

perceptrons = [
    Perceptron(w=np.array([1.0, 0.5]), b=-0.5),
    Perceptron(w=np.array([-1.0, 0.5]), b=-0.5),
    Perceptron(w=np.array([0.0, -1.0]), b=-0.5),
]
# SKIP
def predict_liste(x: np.ndarray, perceptrons: list) -> np.ndarray:
    return np.array([p.output(x) for p in perceptrons], dtype=int)

In [ ]:
# EXPORT
print(predict_liste(np.array([0.0, 0.0]), perceptrons))
print(predict_liste(np.array([0.0, 2.0]), perceptrons))

## Exo 2 — la même chose en matrice `W @ x + b`
Rangez les 3 vecteurs `w` en matrice `W:(3x2)` (une ligne par neurone) et les 3 biais
en vecteur `b:(3,)`. Complétez `predict_vect(x, W, b)` : `h = W @ x + b`,
retourne `(h >= 0).astype(int)` (un Heaviside par neurone, comme en Exo 1).
La cellule de vérification prouve `boucle == matrice` sur tout `X_train`.

In [ ]:
# EXPORT
import numpy as np

# Construisez W (3x2) et b (3,) à partir des 3 perceptrons de l'Exo 1.
# SKIP
def predict_vect(x: np.ndarray, W: np.ndarray, b: np.ndarray) -> np.ndarray:
    h = W @ x + b
    return (h >= 0).astype(int)

W = np.array([[1.0, 0.5], [-1.0, 0.5], [0.0, -1.0]])
b = np.array([-0.5, -0.5, -0.5])

In [ ]:
# EXPORT
for xi in X_train:
    assert np.array_equal(predict_liste(xi, perceptrons), predict_vect(xi, W, b))
print("boucle == matrice OK")
print(predict_vect(np.array([0.0, 0.0]), W, b))
print(predict_vect(np.array([0.0, 2.0]), W, b))

## Constat — pourquoi il faudra `argmax` ?
`[0 0 0]` : aucun neurone ne s'active — quelle classe choisir ?
`[1 1 0]` : deux neurones gagnent — lequel croire ?
3 Heaviside indépendants ne suffisent pas à **décider une classe unique**.
Au TD02, on tranche avec `y = argmax_k h_k` : le score le plus fort gagne.